# Global Terrorism Database: what I found

This is my analysis of the Global Terrorism Database (GTD). Each row is one recorded attack between 1970 and 2017, 181,691 attacks in total. There is nothing for 1993, because the records for that year were lost.

I wanted to answer three questions:

1. Are the number of attacks and the number of casualties related? Are there any strange years or outliers?
2. What are the most common attack methods? Does it change by region or over time?
3. Where did the attacks happen? (a map)

On the way I also looked at how the number of attacks changed over time and which countries were behind it, because I needed that to understand the rest.

## Summary

- The number of attacks went up and down a lot. It was low in the 1970s, rose to about 5,000 a year in the early 1990s, dropped to about 1,000 in 1998, and then shot up after 2011. The highest year was 2014 with 16,903 attacks.
- It wasn't one single trend. Between 1979 and 1992 about a third of all attacks happened in just three countries: Peru, El Salvador and Colombia. After 2010 the big numbers came mostly from Iraq, Pakistan, Afghanistan, India and the Philippines. Iraq had the most attacks of any country in every year from 2013 to 2017.
- Bombings are the most common method (48.6% of attacks), then armed assaults (23.5%) and assassinations (10.6%).
- Years with more attacks also had more casualties (correlation 0.95). A few years don't fit the pattern though. 1998 to 2007 have more casualties than you would expect for the number of attacks, and a few single attacks are huge (the United States in 2001, Japan in 1995, Kenya in 1998).
- On the map, attacks are spread over most of the world. The busiest area by far is Iraq. Other busy areas are Pakistan and Afghanistan, India, the Andes and Central America.

Still to do: I haven't finished checking whether attack methods change by region and over time. The code is in section 6 and I still need to write down what it shows.

Things to be careful with: the way the data was collected changed over time, 1993 is missing, almost half of the attacks (45.6%) have no known group behind them, and a few rows look odd (see the end of the notebook).

## 1. Loading the data

The file is a compressed csv. I use `low_memory=False` so pandas reads the whole file before it guesses the column types. Without it I get a warning about mixed types. I keep `df_raw` as it came and work on a copy called `df`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)

In [ ]:
df_raw = pd.read_csv("globalterrorismdb_0718dist.tar.bz2", compression="bz2", low_memory=False)
df = df_raw.copy()

print(df.shape)

It's a big table: 181,691 rows and 136 columns. I only need about a dozen of the columns. The first column is called `Unnamed: 0`. I'm not sure what it is, but I don't use it.

These are the columns I use:

In [ ]:
cols = ["eventid", "iyear", "imonth", "iday", "country_txt", "region_txt", "city",
        "latitude", "longitude", "attacktype1_txt", "gname", "nkill", "nwound"]
df[cols].head(10)

## 2. A quick look at who did the attacks

`gname` is the name of the group behind the attack.

In [ ]:
top_groups = df["gname"].value_counts().head(10)
pd.DataFrame({"attacks": top_groups, "share_%": (top_groups / len(df) * 100).round(1)})

The biggest group is "Unknown": 82,782 attacks, which is 45.6% of everything. So for almost half of the attacks, nobody knows who did it (or the database doesn't say). After that come the Taliban (7,478), ISIL (5,613), Shining Path (4,555) and the FMLN (3,351).

This matters because I can't say much about groups from this data. I stick to countries, regions, years and methods.

## 3. How did the number of attacks change over time?

In [ ]:
attacks_per_year = df.groupby("iyear").size()

ax = attacks_per_year.plot(kind="bar", figsize=(12, 5), title="Number of attacks by year")
ax.set_xlabel("Year")
ax.set_ylabel("Attacks")
plt.show()

print(attacks_per_year.sort_values(ascending=False).head(5))
print("Is 1993 in the data?", 1993 in attacks_per_year.index)

Things I noticed in the chart:

- It's low in the 1970s and slowly goes up to about 5,000 attacks in 1992.
- 1993 is missing. That is not a real drop to zero. The records for that year were lost and the people who run the database left the year out on purpose, so nobody reads the low number as real.
- There is a big fall in 1998, to about 1,000 attacks. I'm not sure if that is real. From what I read, the data was collected by a different team before and after 1997, so part of the drop could be a change in how attacks were counted. I haven't checked the codebook for this yet.
- After 2011 it goes up very fast and peaks in 2014 at 16,903 attacks. By 2017 it is down to about 11,000.

So I wouldn't just say "terrorism went up and down". The way the data was collected changed too, and that makes the trend harder to read.

## 4. Which countries were behind the changes?

First, the countries with the most attacks in a single year.

In [ ]:
country_year = df.groupby(["iyear", "country_txt"]).size().reset_index(name="attacks")
country_year.sort_values("attacks", ascending=False).head(10)

Iraq in 2014 is the highest, with 3,933 attacks. Iraq has the top spots for 2013 to 2017, and then Pakistan and Afghanistan show up. So the big peak after 2011 is mostly a few countries.

Next, the six countries with the most attacks overall, each on its own chart (the scales are different on purpose, because Iraq is much bigger than the others).

In [ ]:
wide = country_year.pivot(index="iyear", columns="country_txt", values="attacks").fillna(0)

top6 = df["country_txt"].value_counts().head(6).index
wide[top6].plot(subplots=True, layout=(2, 3), figsize=(14, 7), sharex=True, sharey=False)
plt.show()

print(wide[top6].idxmax())

Iraq, Pakistan and Afghanistan are quiet until about 2005 to 2010 and then shoot up. Their peak years are 2014 (Iraq), 2013 (Pakistan) and 2015 (Afghanistan). India and the Philippines go up and down more, and both peak around 2015 or 2016. Colombia looks different. Its highest year is 1997 (about 600 attacks) and it comes down after that.

The top six are picked by the total over all years, which favours the recent years. To see what happened earlier I looked at who had the most attacks between 1979 and 1992, when the first rise happened.

In [ ]:
early = wide.loc[1979:1992].sum().sort_values(ascending=False).head(8)
pd.DataFrame({"attacks": early, "share_%": (early / wide.loc[1979:1992].sum().sum() * 100).round(1)})

In 1979 to 1992 the top countries were Peru (5,762 attacks, 12.1%), El Salvador (5,162, 10.9%) and Colombia (4,383, 9.2%), followed by the United Kingdom, Chile, India, Spain and Nicaragua. The top three alone are about 32% of all attacks in those years.

So the first rise (1970s to early 1990s) and the second one (after 2010) come from different parts of the world. It is two stories, not one trend.

I looked up what was going on in some of these countries. Peru had an internal armed conflict from 1980 to 1994 (about 69,000 people killed), and there were wars in El Salvador and Nicaragua in the 1980s. That fits the numbers, but the data itself doesn't say why, so I can't say these wars caused the attacks in the table. I didn't look up Chile, the UK, Spain or India.

The same table for the recent years is below, so I can compare.

In [ ]:
recent = wide.loc[2012:2017].sum().sort_values(ascending=False).head(8)
pd.DataFrame({"attacks": recent, "share_%": (recent / wide.loc[2012:2017].sum().sum() * 100).round(1)})

## 5. Question 1: are attacks and casualties related?

I define casualties as people killed plus people wounded (`nkill` + `nwound`). Some of these values are empty, so first I checked how many.

In [ ]:
print((df[["nkill", "nwound"]].isna().mean() * 100).round(1))

# an empty value is counted as 0
df["casualties"] = df[["nkill", "nwound"]].sum(axis=1)

by_year = df.groupby("iyear").agg(
    incidents=("casualties", "size"),
    casualties=("casualties", "sum"),
)
by_year["per_incident"] = by_year["casualties"] / by_year["incidents"]
by_year.describe().round(0)

5.7% of `nkill` and 9.0% of `nwound` are empty. I counted those as 0, so the casualty numbers are probably a bit too low.

There are 47 years in the table (1970 to 2017 without 1993). The numbers are very uneven. The median year has 2,870 attacks but the highest has 16,903, and the median year has about 13,700 casualties against a highest of 85,618. A few years are much bigger than the rest, so I plot with log scales.

In [ ]:
print("Pearson: ", round(by_year["incidents"].corr(by_year["casualties"]), 2))
print("Spearman:", round(by_year["incidents"].corr(by_year["casualties"], method="spearman"), 2))

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(by_year["incidents"], by_year["casualties"])
for year, row in by_year.iterrows():
    ax.annotate(year, (row["incidents"], row["casualties"]), fontsize=7)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Attacks per year")
ax.set_ylabel("Casualties per year")
plt.show()

The correlation is strong: 0.95 (Pearson) and 0.83 (Spearman). Years with more attacks have more casualties. This isn't a surprise, because a year's casualties are a total over all of its attacks, so more attacks almost automatically means more casualties. The more interesting part is the years that don't follow the line.

On the plot, 1998 to 2007 sit above the line. They have fewer attacks than other years but more casualties than expected (2001, 2004, 2007 and 1998 stand out most). The early 1970s are in the bottom left corner with few attacks and few casualties. 2012 to 2017 are in the top right and follow the line closely.

Casualties per attack show the same thing:

In [ ]:
print("Lowest:")
print(by_year["per_incident"].sort_values().head(5).round(1))
print()
print("Highest:")
print(by_year["per_incident"].sort_values().tail(5).round(1))

The lowest are 1971 (0.5), 1970 (0.6) and 1977 (0.7). The highest are 2001 (16.0), 2004 (15.2), 1998 (13.8), 2007 (10.9) and 2005 (9.5). So the highest years are all in the 1998 to 2007 block.

One explanation could be that single huge attacks pull a year up. I looked at the biggest single attacks:

In [ ]:
cols_top = ["iyear", "country_txt", "nkill", "nwound", "casualties"]
df.nlargest(10, "casualties")[cols_top]

Things I noticed:

- The first two rows are both the United States in 2001, with almost the same numbers (1,384 killed and 8,190 wounded, and 1,383 and 8,191), and the same total of 9,574. That looks like one event recorded twice, or one event split over two rows. If so, 2001 has about 19,000 casualties counted from what could be one event, and that would explain why 2001 has the highest casualties per attack. I haven't checked this yet. The cell below prints the descriptions of the two rows so I can look.
- Other huge single attacks: Japan in 1995 (5,513), Kenya in 1998 (4,224), Rwanda in 1994 (1,180), Chad in 2008 (1,161), Russia in 2004 (1,071). In 1998 the Kenya attack alone is a big part of the year's total, which helps explain why 1998 looks high.
- The 2014 row for Iraq has 1,570 killed but an empty wounded value, so its casualty count is probably too low.

In [ ]:
df.loc[[73126, 73127], ["eventid", "iyear", "imonth", "iday", "city", "summary"]]

Short answer to question 1: yes, they are strongly correlated, mostly because casualties add up over attacks. The odd years are 1998 to 2007, and a few single attacks are big enough to change a whole year. I would not trust 2001 until I've checked the two rows above.

## 6. Question 2: what are the most common attack methods?

The column `attacktype1_txt` says what kind of attack it was.

In [ ]:
attack_counts = df["attacktype1_txt"].value_counts()
attack_share = (attack_counts / attack_counts.sum() * 100).round(1)
display(pd.DataFrame({"attacks": attack_counts, "share_%": attack_share}))

ax = attack_counts.sort_values().plot(kind="barh", figsize=(9, 5))
ax.bar_label(ax.containers[0], padding=3)
ax.set_xlabel("Attacks")
ax.set_ylabel("")
plt.show()

Bombing/Explosion is the most common method by far: 88,255 attacks, which is 48.6%, almost half. Armed Assault is second (42,669, 23.5%) and Assassination is third (19,312, 10.6%). These three together are about 83% of all attacks.

Now I want to know if the mix changes by region and over time. I show each method as a share of the attacks in that region (or decade), because regions and decades have very different numbers of attacks and I can't compare plain counts.

In [ ]:
top_types = df["attacktype1_txt"].value_counts().head(4).index

by_region = pd.crosstab(df["region_txt"], df["attacktype1_txt"], normalize="index")[top_types] * 100
display(by_region.round(1))

by_region.plot(kind="barh", stacked=True, figsize=(10, 6))
plt.xlabel("% of attacks in that region")
plt.ylabel("")
plt.show()

In [ ]:
df["decade"] = (df["iyear"] // 10) * 10     # 2010 means 2010 to 2017

by_decade = pd.crosstab(df["decade"], df["attacktype1_txt"], normalize="index")[top_types] * 100
display(by_decade.round(1))

by_decade.plot(marker="o", figsize=(9, 5))
plt.ylabel("% of attacks in that decade")
plt.show()

What the two tables show:

- **By region**, bombing is the most common method almost everywhere, and most of all in the Middle East and North Africa (61% of attacks there). Two regions are different: in Central America & the Caribbean and in Sub-Saharan Africa, armed assaults are more common than bombings (42% vs 31%, and 34% vs 32%). Among the bigger regions, Western Europe has the highest share of assassinations (18%).
- **By decade**, the mix changed. Assassinations were 20% of attacks in the 1970s but only about 6% after 2000, while bombings went from 44% to more than half.

## 7. Question 3: where did the attacks happen?

The data has the `latitude` and `longitude` of each attack. First I checked the coordinates.

In [ ]:
geo = df.dropna(subset=["latitude", "longitude"])
print(len(geo), f"({1 - len(geo) / len(df):.1%} of attacks have no coordinates)")
print("Latitudes ok:", geo["latitude"].between(-90, 90).all())
print("Longitudes ok:", geo["longitude"].between(-180, 180).all())

177,134 attacks have coordinates. The other 2.5% can't go on the map, so the map is not 100% complete.

The longitude check says `False`, so some longitude is impossible (it has to be between -180 and 180). When I first plotted it, one point at about -86,000,000 squashed the whole map into a thin line. Here is the row:

In [ ]:
bad = geo[~geo["longitude"].between(-180, 180)]
print(len(bad), "row with an impossible longitude")
bad[["eventid", "iyear", "country_txt", "city", "latitude", "longitude"]]

It is a 1982 attack in Nicaragua (Valle El Oregano, latitude 12.64) with a longitude of -86,185,896. My guess is that the decimal point is missing. -86.185896 is a normal longitude for Nicaragua. I fixed it by dividing by a million. That's an assumption, but it is only one row out of 177,134, so it won't change the map either way.

In [ ]:
geo = geo.copy()
geo.loc[bad.index, "longitude"] = geo.loc[bad.index, "longitude"] / 1_000_000
print("Longitudes ok now:", geo["longitude"].between(-180, 180).all())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.scatter(geo["longitude"], geo["latitude"], s=2, alpha=0.15, color="#2a78d6")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Location of attacks, 1970 to 2017")
plt.show()

Now it looks like a world map, because the dots follow the continents. The dense areas are South Asia, the Middle East, the Andes (Colombia and Peru), Central America, Western Europe and parts of Africa. The United States, Canada and Australia are quite empty.

One problem is that the dense areas are all solid blue, so I can't tell a place with thousands of attacks from a place with a few hundred. So I also made a density map, where the colour shows how many attacks are in each small cell. The colour scale is logarithmic, so each darker step is about ten times more attacks.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
hb = ax.hexbin(geo["longitude"], geo["latitude"], gridsize=120, bins="log", cmap="YlOrRd", mincnt=1)
fig.colorbar(hb, label="Attacks per cell (log scale)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Density of attacks, 1970 to 2017")
plt.show()

This one is much clearer. The darkest area is Iraq, and its cell goes up to about 10,000 attacks, far above anywhere else. There are also dark areas in Pakistan and Afghanistan, northern India, along the Andes, in Central America and in a few places in Europe and southern Africa. Most of the United States, Canada and Australia only have a few attacks per cell.

This fits the earlier sections: Latin America was busy in the 1980s and 90s, and the Middle East and South Asia after 2010. Because the map adds up all the years, it can't show that change by itself.

Short answer to question 3: attacks happened in most parts of the world, but they are very concentrated, and Iraq is the busiest area by far.

## 8. Things to be careful with

- **Collection changed.** The data was collected by different teams in different periods, with different sources. So some of the changes in the numbers (for example the drop in 1998 and part of the rise later) might come from counting differently, not from real changes.
- **1993 is missing.** The records were lost. I can't say anything about that year.
- **Empty values.** 5.7% of `nkill` and 9.0% of `nwound` are empty and I counted them as 0, so casualties are probably too low. 2.5% of attacks have no coordinates.
- **Possible double count.** The two United States rows in 2001 (9,574 casualties each) need checking.
- **Unknown groups.** 45.6% of attacks have no known group.
- **No causes.** I can see where and when attacks happened, but not why. The things I read about conflicts in Peru, El Salvador and Nicaragua fit the numbers, but they don't prove anything.
- **Only counts.** I counted attacks. Attacks differ a lot in size, so more attacks doesn't always mean more harm.

## 9. What I would do next

- Finish section 6 and write down how methods change by region and decade.
- Check the two 2001 rows and the 1998 drop in the codebook.
- Find what is special about the 1998 to 2007 block, for example by looking at attacks per year with a few huge attacks taken out.
- Check Chile, the UK, Spain and India in the 1980s, which I haven't looked up.